In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("disease_dataset.csv")
df.head()

In [ ]:
print(df.info())
print(df.isnull().sum())

In [ ]:
le = LabelEncoder()

df["Gender"] = le.fit_transform(df["Gender"])
df["Medical_History"] = le.fit_transform(df["Medical_History"])
df["Disease"] = le.fit_transform(df["Disease"])

df.head()

In [ ]:
sns.countplot(x='Disease', data=df)
plt.title("Disease Distribution")
plt.show()

sns.countplot(x='Gender', hue='Disease', data=df)
plt.title("Gender vs Disease")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
X = df.drop(["Disease", "Patient_ID"], axis=1)
y = df["Disease"]

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True)
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

In [ ]:
results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print("Model:", name)
    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

In [ ]:
plt.bar(results.keys(), results.values())
plt.xticks(rotation=30)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
best_model_name = max(results, key=results.get)
print("Best Model:", best_model_name)

best_model = trained_models[best_model_name]

In [ ]:
cv_scores = cross_val_score(best_model, X_scaled, y, cv=5)

print("Cross Validation Scores:", cv_scores)
print("Average CV Score:", cv_scores.mean())

In [ ]:
if hasattr(best_model, "predict_proba"):
    y_prob = best_model.predict_proba(X_test)[:,1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    plt.plot(fpr, tpr)
    plt.title("ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.show()

In [ ]:
if hasattr(best_model, "feature_importances_"):
    features = X.columns
    importance = best_model.feature_importances_

    plt.barh(features, importance)
    plt.title("Feature Importance")
    plt.show()
else:
    print("Feature importance not available")

In [ ]:
def predict_disease(age, gender, fever, cough, headache, bp, sugar, chol, history):
    input_data = np.array([[age, gender, fever, cough, headache, bp, sugar, chol, history]])
    input_scaled = scaler.transform(input_data)

    prediction = best_model.predict(input_scaled)[0]

    return "Heart Disease" if prediction == 1 else "Flu"

In [ ]:
def risk_level(sugar, chol):
    if sugar > 180 or chol > 240:
        return "High Risk"
    elif sugar > 140:
        return "Medium Risk"
    else:
        return "Low Risk"

In [ ]:
result = predict_disease(50, 1, 0, 1, 1, 150, 180, 240, 1)
print("Prediction:", result)
print("Risk:", risk_level(180, 240))

In [ ]:
import pickle

pickle.dump(best_model, open("disease_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

In [ ]:
age = int(input("Enter Age: "))
gender = int(input("Gender (0=Female, 1=Male): "))
fever = int(input("Fever (0/1): "))
cough = int(input("Cough (0/1): "))
headache = int(input("Headache (0/1): "))
bp = int(input("BP: "))
sugar = int(input("Sugar: "))
chol = int(input("Cholesterol: "))
history = int(input("Medical History encoded value: "))

prediction = predict_disease(age, gender, fever, cough, headache, bp, sugar, chol, history)
risk = risk_level(sugar, chol)

print("Predicted Disease:", prediction)
print("Risk Level:", risk)

In [ ]:
# Simple UI using widgets
import ipywidgets as widgets
from IPython.display import display

# Input fields
age = widgets.IntText(description="Age")
gender = widgets.Dropdown(options=[("Female",0), ("Male",1)], description="Gender")
fever = widgets.Dropdown(options=[("No",0), ("Yes",1)], description="Fever")
cough = widgets.Dropdown(options=[("No",0), ("Yes",1)], description="Cough")
headache = widgets.Dropdown(options=[("No",0), ("Yes",1)], description="Headache")
bp = widgets.IntText(description="BP")
sugar = widgets.IntText(description="Sugar")
chol = widgets.IntText(description="Cholesterol")
history = widgets.IntText(description="History")

button = widgets.Button(description="Predict")

output = widgets.Output()

# Function
def on_button_click(b):
    with output:
        output.clear_output()

        pred = predict_disease(
            age.value, gender.value, fever.value,
            cough.value, headache.value,
            bp.value, sugar.value, chol.value, history.value
        )

        risk = risk_level(sugar.value, chol.value)

        print("🩺 Predicted Disease:", pred)
        print("⚠️ Risk Level:", risk)

button.on_click(on_button_click)

# Display UI
display(age, gender, fever, cough, headache, bp, sugar, chol, history, button, output)